# Task 1: Data Modeling & Ledger Specification
## **Mana & Coins Game Economy Architecture**
---
### **Executive Guideline for Reviewers & Users**
This notebook serves as the authoritative technical specification and mock dataset generator for backend engineering and data platform teams.

**Key Highlights:**
1. **Zero-Touch Execution:** Click **Run All** (or run cells sequentially). All mathematical curves, sample ledger events, and auxiliary tables will be generated and saved automatically to the local folder.
2. **Single Append-Only Ledger:** Demonstrates how all currency mutations (`MANA`, `COIN`, `FIAT_USD`) are unified into one immutable table without `UPDATE` or `DELETE` statements.
3. **Zero-Trust Triangulation:** Implements strict idempotency across three event sources for real-money purchases: *Client Callback*, *S2S Payment Gateway Webhook*, and *Cron Reconciliation Audit*.

## 1. Mathematical Formulas & Operator Precedence Statement

> ### 📌 **Mathematical Precedence Assumption & Clarification**
> In accordance with standard algebraic operator precedence (PEMDAS / BODMAS), exponentiation binds **strictly and directly to the level variable** ($L, L_C, L_{CS}$) before multiplication by the base coefficient ($3$):
>
> $$\text{Action Cost} = 3 	imes L^c \equiv 3 \cdot (L^c)$$
>
> **Rationale:**  
> The prompt explicitly specifies that the initial action cost at Level 1 is **3 Mana**.  
> If the exponent were applied to the coefficient as $(3 \times L)^c$, then at Level 1 the cost would be $3^{1.1} \approx 3.35$, violating the initial baseline condition. Applying the exponent solely to $L$ guarantees that at $L=1$, $3 \times 1^{1.1} = 3.0$ exactly.
>
> The same strict precedence applies across all game formulas:
- **Action Mana Cost ($m_a$):** $m_a = \text{round}(3 \times L^{1.1})$
- **Charging Speed Mana/Hour:** $(3 \times L_{CS}^{1.1}) \times 1.5$
- **Speed Upgrade Cost (Coins):** $3 \times L_{CS}^{1.1}$
- **Mana Capacity Limit:** $(3 \times L_C^{1.1}) \times 30$
- **Capacity Upgrade Cost (Coins):** $3 \times L_C^{2c} = 3 \times L_C^{2.2}$
- **Coins Per Dollar (CPD):** $\text{CPD} = 4.32 \times m_a$
- **Coins Awarded for Fiat Item:** $\text{USD Price} \times \text{CPD}$

In [ ]:
import json
import os
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta
from typing import List, Optional, Dict, Any
import pandas as pd

class EconomyFormulas:
    """Encapsulates all exponential progression and inflation curves."""
    C_INFLATION: float = 1.1   # c = 1.1
    C2_CAPACITY: float = 30.0  # c2 = 30
    C3_CPD: float = 4.32       # c3 = 4.32
    C4_SPEED: float = 1.5      # c4 = 1.5

    @classmethod
    def action_mana_cost(cls, level: int) -> int:
        """m_a = round(3 * L ^ c)"""
        return round(3.0 * (level ** cls.C_INFLATION))

    @classmethod
    def capacity_limit(cls, level_capacity: int) -> int:
        """Capacity = (3 * L_C ^ c) * c2"""
        return round((3.0 * (level_capacity ** cls.C_INFLATION)) * cls.C2_CAPACITY)

    @classmethod
    def capacity_upgrade_cost(cls, level_capacity: int) -> int:
        """Cost = 3 * L_C ^ (2c) coins"""
        return round(3.0 * (level_capacity ** (2.0 * cls.C_INFLATION)))

    @classmethod
    def speed_mana_per_hour(cls, level_speed: int) -> float:
        """Mana/hour = (3 * L_CS ^ c) * c4"""
        return round((3.0 * (level_speed ** cls.C_INFLATION)) * cls.C4_SPEED, 2)

    @classmethod
    def speed_upgrade_cost(cls, level_speed: int) -> int:
        """Cost = 3 * L_CS ^ c coins"""
        return round(3.0 * (level_speed ** cls.C_INFLATION))

    @classmethod
    def coins_per_dollar(cls, level: int) -> float:
        """CPD = c3 * m_a"""
        m_a = cls.action_mana_cost(level)
        return round(cls.C3_CPD * m_a, 2)

    @classmethod
    def bundle_coins(cls, usd_price: float, level: int) -> int:
        """Coins awarded = usd_price * CPD"""
        return round(usd_price * cls.coins_per_dollar(level))

# Precedence Validation Proofs
print(f"-> Baseline Level 1: Action Cost = {EconomyFormulas.action_mana_cost(1)} Mana (Expected: 3)")
print(f"-> Baseline Level 1: Capacity    = {EconomyFormulas.capacity_limit(1)} Mana (Expected: 90)")
print(f"-> Mid-Game Level 5: Action Cost = {EconomyFormulas.action_mana_cost(5)} Mana | CPD = {EconomyFormulas.coins_per_dollar(5)}")
print(f"-> Late-Game Level 10: Action Cost = {EconomyFormulas.action_mana_cost(10)} Mana | CPD = {EconomyFormulas.coins_per_dollar(10)}")

## 2. Immutable Ledger Event Emitter

The transaction table is modeled with strict event-sourcing principles:
- Every row represents an immutable event with a signed delta (`amount`).
- Running balances are derived dynamically from settled entries.
- Unverified client-side reports are isolated with `amount = 0`.

In [ ]:
@dataclass
class TransactionRecord:
    transaction_id: str
    user_id: str
    timestamp: str
    currency_type: str
    amount: float
    event_type: str
    user_level: int
    capacity_level: int
    speed_level: int
    source: str
    verification_status: str
    external_txn_id: Optional[str]
    idempotency_key: str
    metadata_json: str

class LedgerSimulator:
    """Simulates player lifecycles and produces append-only ledger events."""
    def __init__(self, start_time: datetime):
        self.current_time = start_time
        self.tx_counter = 1000
        self.records: List[TransactionRecord] = []
        self.balances: Dict[str, Dict[str, float]] = {}

    def _next_tx_id(self) -> str:
        self.tx_counter += 1
        return f"tx_{self.tx_counter}"

    def emit(
        self, user_id: str, minutes_offset: int, currency_type: str, amount: float,
        event_type: str, user_level: int, capacity_level: int, speed_level: int,
        source: str, verification_status: str, idempotency_key: str,
        external_txn_id: Optional[str] = None, metadata: Optional[Dict[str, Any]] = None
    ) -> TransactionRecord:
        self.current_time += timedelta(minutes=minutes_offset)
        user_bal = self.balances.setdefault(user_id, {"MANA": 0.0, "COIN": 0.0, "FIAT_USD": 0.0})
        
        if verification_status == "SETTLED" and currency_type in user_bal:
            user_bal[currency_type] += amount

        meta = metadata or {}
        if verification_status == "SETTLED" and currency_type in ("MANA", "COIN"):
            meta["balance_after"] = user_bal[currency_type]

        rec = TransactionRecord(
            transaction_id=self._next_tx_id(),
            user_id=user_id,
            timestamp=self.current_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
            currency_type=currency_type,
            amount=amount,
            event_type=event_type,
            user_level=user_level,
            capacity_level=capacity_level,
            speed_level=speed_level,
            source=source,
            verification_status=verification_status,
            external_txn_id=external_txn_id,
            idempotency_key=idempotency_key,
            metadata_json=json.dumps(meta)
        )
        self.records.append(rec)
        return rec

## 3. Lifecycle Scenarios Execution (Data Contract Coverage)

The simulation below creates a deterministic, highly realistic dataset covering:
1. **Starter onboarding:** Welcome coin gift & background recharge up to capacity (90).
2. **Level 1 Actions:** Mana deductions at $m_a = 3$.
3. **Level 5 Progression & Payment Triangulation:**
   - Client claim: Recorded as `PENDING_VERIFICATION` with `amount = 0`.
   - S2S Webhook: Authoritative settlement granting 388 coins.
   - Fiat revenue booking: $4.99 in `FIAT_USD`.
   - Cron reconciliation: Deduplication check marked `DUPLICATE_IGNORED`.
4. **Economy Sinks:** Capacity upgrade, speed upgrade, 2x speed booster, and coin-to-mana conversion.
5. **Fraud Attempt:** Malicious client callback rejected with zero balance leakage.

In [ ]:
sim = LedgerSimulator(start_time=datetime(2026, 10, 1, 8, 0, 0))
u1 = "usr_alex_01"
u2 = "usr_bella_02"

# 1. Level 1 Gameplay
sim.emit(u1, 0, "COIN", 25, "OCCASIONAL_GIFT", 1, 1, 1, "GAME_ENGINE", "SETTLED", "gift_welcome_alex_01", metadata={"note": "Starter welcome gift"})
sim.emit(u1, 10, "MANA", 90, "RECHARGE_TICK", 1, 1, 1, "GAME_ENGINE", "SETTLED", "recharge_u1_init_cap", metadata={"reason": "Auto-recharge to Level 1 capacity (90)"})
sim.emit(u1, 5, "MANA", -3, "ACTION_SPEND", 1, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_001", metadata={"action": "move_forward", "formula": "3 * 1^1.1 = 3"})
sim.emit(u1, 1, "MANA", -3, "ACTION_SPEND", 1, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_002", metadata={"action": "move_forward"})
sim.emit(u1, 64, "MANA", 6, "RECHARGE_TICK", 1, 1, 1, "GAME_ENGINE", "SETTLED", "recharge_u1_002", metadata={"reason": "Auto-recharge tops up back to capacity"})

# 2. Level 5 Gameplay & Payment Triangulation ($4.99 Bag -> 388 Coins)
sim.emit(u1, 160, "MANA", -18, "ACTION_SPEND", 5, 1, 1, "GAME_ENGINE", "SETTLED", "act_u1_010", metadata={"formula": "round(3 * 5^1.1) = 18"})
order_id = "GPA.3391-4820-9912"
# 2a. Client Callback (Untrusted -> amount = 0, PENDING)
sim.emit(u1, 5, "COIN", 0, "IAP_PURCHASE_CLAIM", 5, 1, 1, "CLIENT_CALLBACK", "PENDING_VERIFICATION", f"claim_{order_id}", external_txn_id=order_id, metadata={"sku": "coin_bag", "price_usd": 4.99})
# 2b. PG Webhook (Trusted -> amount = 388, SETTLED)
sim.emit(u1, 0, "COIN", 388, "IAP_PURCHASE", 5, 1, 1, "PG_WEBHOOK", "SETTLED", f"settle_{order_id}", external_txn_id=order_id, metadata={"sku": "coin_bag", "cpd": 77.76, "coins_awarded": 388})
# 2c. Fiat Revenue Booking
sim.emit(u1, 0, "FIAT_USD", 4.99, "IAP_REVENUE", 5, 1, 1, "PG_WEBHOOK", "SETTLED", f"fiat_{order_id}", external_txn_id=order_id, metadata={"gateway": "Google_Play"})
# 2d. Cron Reconciliation Audit (Duplicate -> amount = 0)
sim.emit(u1, 15, "COIN", 0, "RECONCILIATION_AUDIT", 5, 1, 1, "RECON_JOB", "DUPLICATE_IGNORED", f"recon_{order_id}", external_txn_id=order_id, metadata={"status": "ALREADY_SETTLED"})

# 3. Economy Sinks (Upgrades, Boosts, Exchanges)
sim.emit(u1, 5, "COIN", -14, "UPGRADE_CAPACITY", 5, 2, 1, "GAME_ENGINE", "SETTLED", "upg_cap_u1_lv2", metadata={"new_cap": EconomyFormulas.capacity_limit(2)})
sim.emit(u1, 5, "COIN", -6, "UPGRADE_SPEED", 5, 2, 2, "GAME_ENGINE", "SETTLED", "upg_spd_u1_lv2", metadata={"new_speed": EconomyFormulas.speed_mana_per_hour(2)})
sim.emit(u1, 30, "COIN", -20, "BUY_BOOST", 5, 2, 2, "GAME_ENGINE", "SETTLED", "boost_u1_001", metadata={"type": "2X_SPEED", "duration": 5})
sim.emit(u1, 5, "MANA", 2, "RECHARGE_TICK", 5, 2, 2, "GAME_ENGINE", "SETTLED", "recharge_boost_u1_001", metadata={"boost_active": True})
sim.emit(u1, 55, "COIN", -30, "COIN_TO_MANA_EXCHANGE", 5, 2, 2, "GAME_ENGINE", "SETTLED", "c2m_u1_coin", metadata={"mana_acquired": 100})
sim.emit(u1, 0, "MANA", 100, "COIN_TO_MANA_EXCHANGE", 5, 2, 2, "GAME_ENGINE", "SETTLED", "c2m_u1_mana", metadata={"coins_cost": 30})

# 4. Fraud Attempt Detection & Quarantine
ghost_order = "GPA.9999-0000-FAKE"
sim.emit(u2, 60, "COIN", 0, "IAP_PURCHASE_CLAIM", 10, 4, 3, "CLIENT_CALLBACK", "PENDING_VERIFICATION", f"claim_{ghost_order}", external_txn_id=ghost_order, metadata={"sku": "coin_barrel"})
sim.emit(u2, 15, "COIN", 0, "RECONCILIATION_AUDIT", 10, 4, 3, "RECON_JOB", "REJECTED", f"recon_{ghost_order}", external_txn_id=ghost_order, metadata={"error": "PG_404_NOT_FOUND", "fraud_flag": True})

df_ledger = pd.DataFrame([asdict(r) for r in sim.records])
print(f" Successfully emitted {len(df_ledger)} ledger events across all operational scenarios.")

## 4. Visual Inspection of the Single Append-Only Ledger
Below is the core ledger dataset demonstrating all transaction types, sources, and verification states.

In [ ]:
cols_to_display = ["transaction_id", "user_id", "timestamp", "currency_type", "amount", "event_type", "user_level", "source", "verification_status", "external_txn_id"]
display(df_ledger[cols_to_display])

## 5. Auxiliary Dimensions & Progression Matrix (Levels 1 - 20)
These lookup matrices provide engineering and economy design teams with definitive scaling benchmarks.

In [ ]:
curve_rows = []
for l in range(1, 21):
    m_a = EconomyFormulas.action_mana_cost(l)
    curve_rows.append({
        "level": l,
        "action_mana_cost_m_a": m_a,
        "coins_per_dollar_cpd": EconomyFormulas.coins_per_dollar(l),
        "capacity_if_level_c": EconomyFormulas.capacity_limit(l),
        "capacity_upgrade_cost_coins": EconomyFormulas.capacity_upgrade_cost(l),
        "speed_mana_per_hour": EconomyFormulas.speed_mana_per_hour(l),
        "speed_upgrade_cost_coins": EconomyFormulas.speed_upgrade_cost(l),
        "coins_for_4_99_bag": EconomyFormulas.bundle_coins(4.99, l)
    })
df_curve = pd.DataFrame(curve_rows)
display(df_curve.head(10))

# Auxiliary Catalog
df_catalog = pd.DataFrame([
    {"sku_id": "coin_drops", "name": "Drops", "usd_price": 0.99, "tier": 1},
    {"sku_id": "coin_bag", "name": "Bag", "usd_price": 4.99, "tier": 2},
    {"sku_id": "coin_chest", "name": "Chest", "usd_price": 9.99, "tier": 3},
    {"sku_id": "coin_barrel", "name": "Barrel", "usd_price": 19.99, "tier": 4},
])

# Auxiliary User State
df_user_state = pd.DataFrame([
    {
        "user_id": "usr_alex_01",
        "current_level": 5,
        "mana_capacity_level": 2,
        "charging_speed_level": 2,
        "mana_capacity_value": EconomyFormulas.capacity_limit(2),
        "mana_speed_per_hour": EconomyFormulas.speed_mana_per_hour(2),
        "mana_balance": 174,
        "coins_balance": 343,
        "total_usd_spent": 4.99,
        "last_active_at": "2026-10-01T14:00:00Z"
    },
    {
        "user_id": "usr_bella_02",
        "current_level": 10,
        "mana_capacity_level": 4,
        "charging_speed_level": 3,
        "mana_capacity_value": EconomyFormulas.capacity_limit(4),
        "mana_speed_per_hour": EconomyFormulas.speed_mana_per_hour(3),
        "mana_balance": 320,
        "coins_balance": 150,
        "total_usd_spent": 0.00,
        "last_active_at": "2026-10-01T15:15:00Z"
    }
])

## 6. Zero-Touch Export to Production CSV Files
Saves all four artifacts directly into the directory so engineering teams have immediate access to raw CSV data.

In [ ]:
target_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."

df_ledger.to_csv(os.path.join(target_dir, "transactions_sample.csv"), index=False)
df_curve.to_csv(os.path.join(target_dir, "dim_level_economy_curve.csv"), index=False)
df_catalog.to_csv(os.path.join(target_dir, "dim_iap_catalog.csv"), index=False)
df_user_state.to_csv(os.path.join(target_dir, "dim_user_state.csv"), index=False)

print(" [SUCCESS] All four CSV files generated and saved to disk:")
print("  1. transactions_sample.csv")
print("  2. dim_iap_catalog.csv")
print("  3. dim_user_state.csv")
print("  4. dim_level_economy_curve.csv")